# Домашняя работа: Value Iteration и Policy Iteration
Цель работы: реализовать и сравнить два классических алгоритма обучения с подкреплением для решения сред Grid World и Frozen Lake.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import gymnasium as gym
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# --- Общие параметры и константы ---
GAMMA = 0.9 # Коэффициент дисконтирования
ACTIONS_MAP_FL = {0: "←", 1: "↓", 2: "→", 3: "↑"} # Для Frozen Lake
ACTIONS_MAP_GW = {'U': '↑', 'D': '↓', 'L': '←', 'R': '→'} # Для Grid World

## Вспомогательные функции для визуализации (единый блок)
Здесь собраны все функции, необходимые для отрисовки результатов. Запустите эту ячейку один раз.

In [ ]:
def plot_grid_world_v_function(V, grid_size, terminal_states, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    colors = [(1, 0.6, 0.6), (1, 1, 1), (0.6, 0.9, 0.7)]
    cmap = LinearSegmentedColormap.from_list('rg', colors, N=256)
    v_display = np.copy(V)
    sns.heatmap(v_display, annot=True, fmt='.2f', cmap=cmap, ax=ax,
                linewidths=0.5, linecolor='gray', cbar=False,
                xticklabels=range(grid_size), yticklabels=range(grid_size))
    for (r, c), reward in terminal_states.items():
        color = 'green' if reward > 0 else 'red'
        rect = plt.Rectangle([c, r], 1, 1, fill=False, edgecolor=color, linewidth=4)
        ax.add_patch(rect)
    ax.set_title(f'Grid World: Функция ценности (V) - {title}')
    ax.set_xlabel('Столбец')
    ax.set_ylabel('Ряд')
    plt.tight_layout()
    plt.show()


def plot_grid_world_policy(policy, grid_size, terminal_states, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(0, grid_size)
    ax.set_ylim(0, grid_size)
    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.grid(True, color='lightgray')
    ax.set_xticklabels(range(grid_size))
    ax.set_yticklabels(range(grid_size))
    arrow_map = {'U': (0, 0.3), 'D': (0, -0.3), 'L': (-0.3, 0), 'R': (0.3, 0)}
    for r in range(grid_size):
        for c in range(grid_size):
            cx, cy = c + 0.5, (grid_size - 1 - r) + 0.5
            if (r, c) in terminal_states:
                reward = terminal_states[(r, c)]
                label = 'GOAL' if reward > 0 else 'HOLE'
                color = 'green' if reward > 0 else 'red'
                ax.text(cx, cy, label, ha='center', va='center',
                        fontsize=11, fontweight='bold', color=color)
            else:
                action = policy[r, c]
                if action in arrow_map:
                    dx, dy = arrow_map[action]
                    ax.annotate('', xy=(cx + dx, cy + dy), xytext=(cx, cy),
                                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.set_title(f'Grid World: Политика (π) - {title}')
    ax.set_xlabel('Столбец')
    ax.set_ylabel('Ряд')
    plt.tight_layout()
    plt.show()


def plot_frozen_lake_v_function(V, env, grid_size, title):
    desc = env.unwrapped.desc.astype(str)
    fig, ax = plt.subplots(figsize=(7, 6))
    V_grid = V.reshape(grid_size, grid_size)
    colors = [(0.85, 0.3, 0.3), (1, 1, 1), (0.3, 0.8, 0.4)]
    cmap = LinearSegmentedColormap.from_list('rg', colors, N=256)
    im = ax.imshow(V_grid, cmap=cmap, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    for r in range(grid_size):
        for c in range(grid_size):
            cell = desc[r, c]
            v_val = V_grid[r, c]
            ax.text(c, r, f'{v_val:.2f}', ha='center', va='center', fontsize=10, color='black')
            if cell == 'H':
                ax.text(c, r - 0.25, 'HOLE', ha='center', va='center',
                        fontsize=8, fontweight='bold', color='red')
                rect = plt.Rectangle([c - 0.5, r - 0.5], 1, 1, fill=False, edgecolor='red', linewidth=3)
                ax.add_patch(rect)
            elif cell == 'G':
                ax.text(c, r - 0.25, 'GOAL', ha='center', va='center',
                        fontsize=8, fontweight='bold', color='darkgreen')
                rect = plt.Rectangle([c - 0.5, r - 0.5], 1, 1, fill=False, edgecolor='darkgreen', linewidth=3)
                ax.add_patch(rect)
            elif cell == 'S':
                ax.text(c, r - 0.25, 'START', ha='center', va='center',
                        fontsize=8, fontweight='bold', color='blue')
                rect = plt.Rectangle([c - 0.5, r - 0.5], 1, 1, fill=False, edgecolor='blue', linewidth=3)
                ax.add_patch(rect)
    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.set_xticklabels(range(grid_size))
    ax.set_yticklabels(range(grid_size))
    ax.set_title(f'Frozen Lake: Функция ценности (V) - {title}')
    ax.set_xlabel('Столбец')
    ax.set_ylabel('Ряд')
    plt.tight_layout()
    plt.show()


def plot_frozen_lake_policy(policy_indices, env, grid_size, title):
    desc = env.unwrapped.desc.astype(str)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-0.5, grid_size - 0.5)
    ax.set_ylim(-0.5, grid_size - 0.5)
    ax.invert_yaxis()
    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.grid(True, color='lightgray')
    ax.set_xticklabels(range(grid_size))
    ax.set_yticklabels(range(grid_size))
    arrow_dx = {0: -0.3, 1: 0, 2: 0.3, 3: 0}
    arrow_dy = {0: 0, 1: 0.3, 2: 0, 3: -0.3}
    for state in range(grid_size * grid_size):
        r, c = state // grid_size, state % grid_size
        cell = desc[r, c]
        if cell == 'H':
            ax.text(c, r, 'H', ha='center', va='center', fontsize=14, fontweight='bold', color='red')
        elif cell == 'G':
            ax.text(c, r, 'G', ha='center', va='center', fontsize=14, fontweight='bold', color='darkgreen')
        else:
            action = int(policy_indices[state])
            dx = arrow_dx[action]
            dy = arrow_dy[action]
            ax.annotate('', xy=(c + dx, r + dy), xytext=(c, r),
                        arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.set_title(f'Frozen Lake: Политика (π) - {title}')
    ax.set_xlabel('Столбец')
    ax.set_ylabel('Ряд')
    plt.tight_layout()
    plt.show()

# Задание 1: Value Iteration на Grid World
В этом задании вам предстоит реализовать алгоритм Value Iteration (Итерация по ценности). Вы будете работать с простой сеточной средой (Grid World), где агент должен найти оптимальный путь от старта (S) к цели (G), избегая ям (H).

In [ ]:
# --- Параметры среды Grid World ---
GRID_SIZE_GW = 4                    # Размер квадратной сетки: 4x4 клетки
TERMINAL_STATES_GW = {(0, 2): 10, (2, 2): -10}  # Награда при входе в терминал
STEP_REWARD_GW = -1                 # Награда за обычный переход
ACTIONS_GW = {
    'U': (-1, 0),  # Вверх
    'D': (1, 0),   # Вниз
    'L': (0, -1),  # Влево
    'R': (0, 1)    # Вправо
}

## Задача 1.1: Реализация Value Iteration

Допишите код в функции `value_iteration`. Вам нужно реализовать ключевой шаг алгоритма — обновление ценности состояния на основе уравнения Беллмана.

Формула для обновления:

$$V_{k+1}(s) = \max_{a \in A} \sum_{s'} P(s'|s,a) \cdot \left[ R(s,a,s') + \gamma V_k(s') \right]$$

Поскольку среда детерминированная, $P(s'|s,a)=1$ для одного следующего состояния $s'$ и 0 для всех остальных. Формула упрощается до:

$$V_{k+1}(s) = \max_{a \in A} \left( R_a + \gamma V_k(s') \right)$$

**Соглашение о терминалах.** Награда +10 или −10 получается на переходе в терминальную клетку. После этого эпизод завершён, поэтому V(terminal)=0. Значение соседней клетки уже учитывает терминальную награду.

In [ ]:
def value_iteration(verbose=True):
    """
    Выполняет алгоритм итерации по ценности для поиска оптимальной функции ценности.
    Параметры:
        verbose (bool, optional): Если True, выводит отладочную информацию в процессе выполнения.
                                 По умолчанию True.
    Возвращает:
        np.array: Оптимальная функция ценности V*.
    """
    V = np.zeros((GRID_SIZE_GW, GRID_SIZE_GW))
    iteration = 0
    while True:
        V_new = np.copy(V)
        delta = 0

        if verbose:
            print(f"--- VI Итерация: {iteration} ---")
            print(np.round(V, 2))

        for r in range(GRID_SIZE_GW):
            for c in range(GRID_SIZE_GW):
                if (r, c) in TERMINAL_STATES_GW:
                    V_new[r, c] = 0
                    continue

                # Задача 1.1: вычисляем Q(s,a) для каждого действия и берём максимум
                action_values = []
                for action, (dr, dc) in ACTIONS_GW.items():
                    nr, nc = r + dr, c + dc
                    # Если выходим за границы — остаёмся на месте
                    if nr < 0 or nr >= GRID_SIZE_GW or nc < 0 or nc >= GRID_SIZE_GW:
                        nr, nc = r, c
                    if (nr, nc) in TERMINAL_STATES_GW:
                        reward = TERMINAL_STATES_GW[(nr, nc)]
                    else:
                        reward = STEP_REWARD_GW
                    action_values.append(reward + GAMMA * V[nr, nc])

                best_value = max(action_values)

                delta = max(delta, np.abs(best_value - V[r, c]))
                V_new[r, c] = best_value

        V = V_new
        iteration += 1

        if delta < 1e-4:
            print(f"\nСходимость достигнута на итерации {iteration}!")
            break

    return V

## Задача 1.2: Извлечение оптимальной политики

На основе найденной оптимальной функции ценности $V^*$ теперь нужно найти оптимальную политику $\pi^*$. Для каждого состояния $s$ найдите действие $a$, которое максимизирует ожидаемую награду:

$$\pi^*(s) = \arg\max_{a \in A} \left( R_a + \gamma V^*(s') \right)$$

In [ ]:
def extract_policy(V):
    """
    Извлекает оптимальную политику из оптимальной функции ценности.
    Args:
        V (np.array): Оптимальная функция ценности V*.
    Returns:
        np.array: Сетка с оптимальными действиями для каждого состояния.
    """
    policy = np.full((GRID_SIZE_GW, GRID_SIZE_GW), '', dtype=object)
    for r in range(GRID_SIZE_GW):
        for c in range(GRID_SIZE_GW):
            if (r, c) in TERMINAL_STATES_GW:
                continue

            # Задача 1.2: для каждого состояния (r, c) находим лучшее действие
            best_action = None
            best_value = float('-inf')
            for action, (dr, dc) in ACTIONS_GW.items():
                nr, nc = r + dr, c + dc
                if nr < 0 or nr >= GRID_SIZE_GW or nc < 0 or nc >= GRID_SIZE_GW:
                    nr, nc = r, c
                if (nr, nc) in TERMINAL_STATES_GW:
                    reward = TERMINAL_STATES_GW[(nr, nc)]
                else:
                    reward = STEP_REWARD_GW
                q_value = reward + GAMMA * V[nr, nc]
                if q_value > best_value:
                    best_value = q_value
                    best_action = action

            policy[r, c] = best_action

    return policy

## Проверка Задания 1
Value Iteration — детерминированный алгоритм динамического программирования. Для заданного MDP он сходится к единственной оптимальной функции ценности V*. Проверяем результат с помощью прямых `assert`-ов.

In [ ]:
# Запускаем алгоритмы
V_star_gw = value_iteration(verbose=False)
pi_star_gw = extract_policy(V_star_gw)

# Тесты для проверки
assert pi_star_gw[1, 1] in ['U','R'], f"Ошибка в политике для [1,1]: ожидалось 'U' или 'R', получено {pi_star_gw[1,1]}"
assert pi_star_gw[0, 1] == 'R', f"Ошибка в политике для [0,1]: ожидалось 'R', получено {pi_star_gw[0,1]}"
assert pi_star_gw[2, 0] == 'U', f"Ошибка в политике для [2,0]: ожидалось 'U', получено {pi_star_gw[2,0]}"

assert V_star_gw[0, 2] == 0 and V_star_gw[2, 2] == 0, "После входа в терминал будущая ценность равна 0."
assert np.isclose(V_star_gw[0, 1], 10), "Сосед цели должен учитывать награду +10 за вход."

print("\nПоздравляем! Задание 1 выполнено корректно.")

## Визуализация для Задания 1: Value Iteration

In [ ]:
print("Результаты для Grid World (Value Iteration):")
plot_grid_world_v_function(V_star_gw, GRID_SIZE_GW, TERMINAL_STATES_GW, "Оптимальная V*")
plot_grid_world_policy(pi_star_gw, GRID_SIZE_GW, TERMINAL_STATES_GW, "Оптимальная π*")

# Задание 2: Policy Iteration на Frozen Lake (детерминированная среда)

Теперь вы реализуете Policy Iteration (Итерация по политике). Этот алгоритм состоит из двух чередующихся шагов:

1. **Policy Evaluation**: Оценка текущей политики (вычисление $V^\pi(s)$).
2. **Policy Improvement**: Улучшение политики на основе вычисленной функции ценности.

Мы будем использовать среду `FrozenLake-v1` из библиотеки Gymnasium. Сначала рассмотрим её нескользкий (детерминированный) вариант: `is_slippery=False`.

In [ ]:
# --- Параметры для Policy Iteration ---
THETA_POLICY_EVAL = 1e-10 # Порог для сходимости оценки функции ценности

## Задача 2.1: Оценка политики (Policy Evaluation)

Допишите код в функции `compute_value_function`. На этом шаге для заданной политики $\pi$ вычисляется функция ценности $V^\pi(s)$ путём итеративного применения уравнения Беллмана для ожидания:

$$V^\pi_{k+1}(s) = \sum_{s'} P(s'|s,\pi(s)) \cdot \left[ R(s,a,s') + \gamma V^\pi_k(s') \right]$$

Цикл продолжается до тех пор, пока максимальное изменение ценности состояний ($\|V^{k+1}-V^k\|_\infty$) не станет меньше `theta`.

In [ ]:
def compute_value_function(policy_indices, env, gamma, theta_eval):
    """
    Оценивает заданную политику (синхронный итеративный вариант).
    Args:
        policy_indices (np.array): 1D массив, где для каждого состояния указан индекс действия.
        env (gym.Env): Среда Gymnasium.
        gamma (float): Коэффициент дисконтирования.
        theta_eval (float): Порог для определения сходимости.
    Returns:
        np.array: Функция ценности V для данной политики.
    """
    V = np.zeros(env.observation_space.n)
    while True:
        V_old = np.copy(V)
        delta = 0
        for state in range(env.observation_space.n):
            action = int(policy_indices[state])
            new_v_s = 0
            # Задача 2.1: уравнение Беллмана для ожидания
            # Используем V_old (синхронное обновление)
            # done=True означает терминальное состояние — будущей ценности нет
            for prob, next_state, reward, done in env.unwrapped.P[state][action]:
                new_v_s += prob * (reward + gamma * V_old[next_state] * (not done))
            V[state] = new_v_s
            delta = max(delta, np.abs(V_old[state] - V[state]))
        if delta < theta_eval:
            break
    return V

## Задача 2.2: Улучшение политики (Policy Improvement)

Теперь, имея функцию ценности $V^\pi$, улучшите политику, сделав её «жадной» по отношению к $V^\pi$. Для каждого состояния $s$ выберите действие, которое максимизирует ожидаемую награду:

$$\pi'(s) = \arg\max_{a \in A} Q^\pi(s,a) = \arg\max_{a \in A} \sum_{s'} P(s'|s,a) \cdot \left[ R(s,a,s') + \gamma V^\pi(s') \right]$$

In [ ]:
def extract_policy_from_v(value_table, env, gamma):
    """
    Извлекает новую детерминированную политику (1D массив индексов действий)
    на основе функции ценности V.
    Args:
        value_table (np.array): Функция ценности V.
        env (gym.Env): Среда Gymnasium.
        gamma (float): Коэффициент дисконтирования.
    Returns:
        np.array: Новая политика (массив индексов действий).
    """
    policy_indices = np.zeros(env.observation_space.n, dtype=int)
    for state in range(env.observation_space.n):
        Q_table = np.zeros(env.action_space.n)
        # Задача 2.2: вычисляем Q(s,a) для каждого действия
        for action in range(env.action_space.n):
            for prob, next_state, reward, done in env.unwrapped.P[state][action]:
                Q_table[action] += prob * (reward + gamma * value_table[next_state] * (not done))
        policy_indices[state] = np.argmax(Q_table)
    return policy_indices

## Задача 2.3: Основной цикл Policy Iteration

Объедините шаги оценки и улучшения в один цикл. Алгоритм сходится, когда политика перестаёт изменяться после шага улучшения.

In [ ]:
def policy_iteration(env, gamma, theta_eval, verbose=True):
    """
    Основной алгоритм Policy Iteration.
    Args:
        env (gym.Env): Среда Gymnasium.
        gamma (float): Коэффициент дисконтирования.
        theta_eval (float): Порог для сходимости Policy Evaluation.
        verbose (bool, optional): Если True, выводит отладочную информацию. По умолчанию True.
    Returns:
        (np.array, np.array): Оптимальная политика и оптимальная функция ценности.
    """
    rng = np.random.default_rng(42)
    old_policy_indices = rng.integers(0, env.action_space.n, env.observation_space.n)
    no_of_iterations = 200
    V = np.zeros(env.observation_space.n)
    new_policy_indices = old_policy_indices.copy()

    for i in range(no_of_iterations):
        # Задача 2.3: основной цикл Policy Iteration

        # 1. Policy Evaluation — оцениваем текущую политику
        V = compute_value_function(old_policy_indices, env, gamma, theta_eval)

        # 2. Policy Improvement — улучшаем политику жадно
        new_policy_indices = extract_policy_from_v(V, env, gamma)

        if verbose:
            print(f"--- PI Итерация {i+1} ---")
            print("Функция ценности:")
            grid_size = int(np.sqrt(env.observation_space.n))
            print(np.round(V.reshape(grid_size, grid_size), 3))
            print("Политика:")
            print(new_policy_indices.reshape(grid_size, grid_size))

        # 3. Проверка сходимости — политика не изменилась?
        if np.array_equal(old_policy_indices, new_policy_indices):
            if verbose:
                print(f"Политика сошлась на итерации {i+1}!")
            break

        # 4. Обновляем политику
        old_policy_indices = new_policy_indices.copy()

    return new_policy_indices, V

## Проверка Задания 2
Policy Iteration сходится к оптимальной политике. Проверяем финальный результат.

In [ ]:
# Создаем детерминированную среду
env_no_slip = gym.make('FrozenLake-v1', is_slippery=False, render_mode=None)
GRID_SIZE_FL = int(np.sqrt(env_no_slip.observation_space.n))

# --- Тесты для отдельных функций ---
test_policy_eval = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 0])
V_eval_test = compute_value_function(test_policy_eval, env_no_slip, gamma=0.9, theta_eval=1e-10)
assert np.isclose(V_eval_test[0], 0., atol=0.01), f"Ошибка в compute_value_function. V[0] = {V_eval_test[0]:.2f}, ожидалось ~0."
assert np.isclose(V_eval_test[14], 1, atol=0.01), f"Ошибка в compute_value_function. V[14] = {V_eval_test[14]:.2f}, ожидалось ~1"
print("Функция compute_value_function прошла тест.")

policy_impr_test = extract_policy_from_v(V_eval_test, env_no_slip, gamma=0.9)
assert policy_impr_test[6] == 1 and policy_impr_test[1] == 2, "Ошибка в extract_policy_from_v."
print("Функция extract_policy_from_v прошла тест.")

# --- Тест для всего алгоритма Policy Iteration ---
optimal_policy_no_slip, V_optimal_no_slip = policy_iteration(env_no_slip, gamma=0.9, theta_eval=1e-10, verbose=False)
assert optimal_policy_no_slip[10] == 1, "Финальная политика в состоянии 10 для is_slippery=False неверна."
assert np.isclose(V_optimal_no_slip[10], 0.9, atol=0.01), f"Финальная V[10] для is_slippery=False неверна: {V_optimal_no_slip[10]:.2f}, ожидалось ~0.9"

print("\nПоздравляем! Задание 2 выполнено корректно.")
env_no_slip.close()

## Визуализация для Задания 2: Policy Iteration (Нескользкая)

In [ ]:
env_no_slip = gym.make('FrozenLake-v1', is_slippery=False, render_mode=None)
print("Результаты для Frozen Lake (is_slippery=False):")
plot_frozen_lake_v_function(V_optimal_no_slip, env_no_slip, GRID_SIZE_FL, "Оптимальная V* (Нескользкая)")
plot_frozen_lake_policy(optimal_policy_no_slip, env_no_slip, GRID_SIZE_FL, "Оптимальная π* (Нескользкая)")
env_no_slip.close()

# Задание 3: Policy Iteration на Frozen Lake (стохастическая среда)

Теперь применим тот же самый код к скользкой (стохастической) версии `FrozenLake-v1`. В этой среде действия агента не всегда приводят к ожидаемому результату. Например, если агент выбирает идти «вверх», он с некоторой вероятностью может сдвинуться влево или вправо.

In [ ]:
env_slip = gym.make('FrozenLake-v1', is_slippery=True, render_mode=None)
optimal_policy_slip, V_optimal_slip = policy_iteration(
    env_slip, gamma=0.9, theta_eval=1e-10, verbose=False
)

print(f"V(S_14) = {V_optimal_slip[14]:.4f}")
assert np.isclose(V_optimal_slip[14], 0.64, atol=0.05), \
    f"V[14] неверно: {V_optimal_slip[14]:.2f}, ожидалось ~0.64"
print("Поздравляем! Задание 3 выполнено корректно.")
env_slip.close()

## Визуализация для Задания 3: Policy Iteration (Скользкая)

Обратите внимание на, казалось бы, странные ходы. В некоторых клетках правильная политика может предписывать идти в стену, например, в положении между дыркой и стеной. Почему? Потому что на «скользком льду» это самый безопасный ход: с вероятностью 1/3 агент попадёт в стену, и с вероятностью 1/3 его снесёт в каждую из перпендикулярных сторон, но при этом он точно не попадёт в дырку.

In [ ]:
env_slip_vis = gym.make('FrozenLake-v1', is_slippery=True, render_mode=None)

print("\nРезультаты для Frozen Lake (is_slippery=True):")
plot_frozen_lake_v_function(V_optimal_slip, env_slip_vis, GRID_SIZE_FL, "Оптимальная V* (Скользкая)")
plot_frozen_lake_policy(optimal_policy_slip, env_slip_vis, GRID_SIZE_FL, "Оптимальная π* (Скользкая)")
env_slip_vis.close()